# CNN for Cat and Dog Classification (Keras)
Prepare train/validation/test data, train a CNN for 10 epochs, evaluate it, and visualize predictions.

In [ ]:
import os
import shutil
import random
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.utils import image_dataset_from_directory

random.seed(42)
tf.random.set_seed(42)
print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

## 1. Unzip the dataset

In [ ]:
zip_path = "data_from_kaggle.zip"
source_path = "data_from_kaggle"
output_path = "dataset"

if not os.path.exists(source_path):  # unzip only once
    shutil.unpack_archive(zip_path)
print(os.listdir(source_path))

## 2. Prepare train / validation / test folders
`data_from_kaggle/train` is split 80/20 into train and validation. `data_from_kaggle/test` is the test set.

In [ ]:
train_ratio = 0.8
train_source = os.path.join(source_path, "train")
test_source = os.path.join(source_path, "test")

def list_images(folder, cls):
    return sorted(f for f in os.listdir(folder) if f.lower().startswith(cls))

cat_imgs = list_images(train_source, "cat")
dog_imgs = list_images(train_source, "dog")
random.shuffle(cat_imgs)
random.shuffle(dog_imgs)

cat_split = int(len(cat_imgs) * train_ratio)
dog_split = int(len(dog_imgs) * train_ratio)

splits = {
    ("train", "cat"): (train_source, cat_imgs[:cat_split]),
    ("train", "dog"): (train_source, dog_imgs[:dog_split]),
    ("validation", "cat"): (train_source, cat_imgs[cat_split:]),
    ("validation", "dog"): (train_source, dog_imgs[dog_split:]),
    ("test", "cat"): (test_source, list_images(test_source, "cat")),
    ("test", "dog"): (test_source, list_images(test_source, "dog")),
}

In [ ]:
if os.path.exists(output_path):  # rebuild cleanly on every run
    shutil.rmtree(output_path)

for (split, cls), (src, files) in splits.items():
    dst = os.path.join(output_path, split, cls)
    os.makedirs(dst, exist_ok=True)
    for f in files:
        shutil.copy(os.path.join(src, f), os.path.join(dst, f))

for split in ["train", "validation", "test"]:
    n_cat = len(os.listdir(os.path.join(output_path, split, "cat")))
    n_dog = len(os.listdir(os.path.join(output_path, split, "dog")))
    print(f"{split:10s} cat: {n_cat:5d}  dog: {n_dog:5d}  total: {n_cat + n_dog}")

## 3. Load the datasets

In [ ]:
IMG_SIZE = (200, 180)
BATCH_SIZE = 32
EPOCHS = 10

train_dataset = image_dataset_from_directory(os.path.join(output_path, "train"), image_size=IMG_SIZE,
                                             batch_size=BATCH_SIZE, label_mode="binary", shuffle=True, seed=42)
val_dataset = image_dataset_from_directory(os.path.join(output_path, "validation"), image_size=IMG_SIZE,
                                           batch_size=BATCH_SIZE, label_mode="binary", shuffle=False)
test_dataset = image_dataset_from_directory(os.path.join(output_path, "test"), image_size=IMG_SIZE,
                                            batch_size=BATCH_SIZE, label_mode="binary", shuffle=False)

class_names = train_dataset.class_names
print("Classes:", class_names)  # cat = 0, dog = 1

# prefetch so the GPU is not waiting on disk
AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset.prefetch(AUTOTUNE)
val_dataset = val_dataset.prefetch(AUTOTUNE)
test_dataset = test_dataset.prefetch(AUTOTUNE)

In [ ]:
plt.figure(figsize=(15, 3))
for images, labels in train_dataset.take(1):
    for i in range(5):
        plt.subplot(1, 5, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[int(labels[i][0])])
        plt.axis("off")
plt.show()

## 4. Build the CNN

In [ ]:
def build_model():
    inputs = keras.Input(shape=IMG_SIZE + (3,))
    x = layers.Rescaling(1.0 / 255)(inputs)  # pixels to [0, 1]
    x = layers.Conv2D(32, 3, activation="relu")(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Conv2D(64, 3, activation="relu")(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Conv2D(128, 3, activation="relu")(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Conv2D(256, 3, activation="relu")(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Conv2D(256, 3, activation="relu")(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Flatten()(x)
    x = layers.Dropout(0.5)(x)  # reduce overfitting
    outputs = layers.Dense(1, activation="sigmoid")(x)  # probability of dog
    return keras.Model(inputs, outputs)

model = build_model()
model.compile(optimizer="rmsprop", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

## 5. Train (epochs = 10)

In [ ]:
history = model.fit(train_dataset, epochs=EPOCHS, validation_data=val_dataset, verbose=2)

In [ ]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history["loss"], marker="o", label="Training loss")
plt.plot(history.history["val_loss"], marker="o", label="Validation loss")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("Loss"); plt.legend(); plt.grid()
plt.subplot(1, 2, 2)
plt.plot(history.history["accuracy"], marker="o", label="Training acc")
plt.plot(history.history["val_accuracy"], marker="o", label="Validation acc")
plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.title("Accuracy"); plt.legend(); plt.grid()
plt.show()

## 6. Evaluate on validation and test sets

In [ ]:
val_loss, val_acc = model.evaluate(val_dataset, verbose=0)
test_loss, test_acc = model.evaluate(test_dataset, verbose=0)
print(f"Validation loss: {val_loss:.4f}  accuracy: {val_acc:.4f}")
print(f"Test loss:       {test_loss:.4f}  accuracy: {test_acc:.4f}")

In [ ]:
# confusion matrix on the test set
y_true = np.concatenate([y.numpy().flatten() for _, y in test_dataset]).astype(int)
y_prob = model.predict(test_dataset, verbose=0).flatten()
y_pred = (y_prob > 0.5).astype(int)

cm = tf.math.confusion_matrix(y_true, y_pred).numpy()
print("Rows = true, columns = predicted", class_names)
print(cm)
print(f"Test accuracy (manual): {np.mean(y_pred == y_true):.4f}")

## 7. Prediction examples (predicted vs. true)

In [ ]:
# take a shuffled batch so both cats and dogs appear
sample_ds = image_dataset_from_directory(os.path.join(output_path, "test"), image_size=IMG_SIZE,
                                         batch_size=12, label_mode="binary", shuffle=True, seed=7)
images, labels = next(iter(sample_ds))
probs = model.predict(images, verbose=0).flatten()

plt.figure(figsize=(16, 10))
for i in range(12):
    plt.subplot(3, 4, i + 1)
    plt.imshow(images[i].numpy().astype("uint8"))
    true_c = class_names[int(labels[i][0])]
    pred_c = class_names[int(probs[i] > 0.5)]
    color = "green" if true_c == pred_c else "red"
    plt.title(f"True: {true_c} | Pred: {pred_c} ({probs[i]:.2f})", color=color)
    plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
model.save("cnn_cat_dog.keras")
print("Model saved to cnn_cat_dog.keras")

## 8. Conclusion
- Training loss falls and accuracy rises over 10 epochs, so training works.
- Validation and test accuracy above are measured on images the model never trained on, so evaluation works.
- Green titles are correct predictions, red titles are mistakes.